In [1]:
import optuna
import axelrod as axl
from axelrod.action import Action, actions_to_str
from axelrod.player import Player
from axelrod.strategy_transformers import (
    FinalTransformer,
    TrackHistoryTransformer,
)
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import numpy as np
from collections import Counter
import pandas as pd
from math import exp
import numpy as np


c:\Users\Ognjen\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
C, D = Action.C, Action.D

class FuzzyMethods():
    @staticmethod
    def sigmoid(x, center, scale=10):
        return 1 / (1 + exp(-scale * (x - center)))
    
    @staticmethod
    def fuzzy_gate(mu_D, d_thresh, mu_C, c_thresh):
        d_condition = FuzzyMethods.sigmoid(mu_D, center=d_thresh)
        c_condition = 1 - FuzzyMethods.sigmoid(mu_C, center=c_thresh)

        w1 = d_condition * c_condition

        return w1
    
    @staticmethod
    def calc_cooperation(self, opponent):
        return (Counter(opponent.history)[C])/len(opponent.history)*100
    
    @staticmethod
    def calc_adaptivity(self, opponent):
        adapCounter = 0
        adapReaction = 0

        if(len(self.history) < 3): 
            return 0
        
        for i in range(3, len(self.history)):
            if (self.history[i-3] == C and self.history[i-2] == D):
                adapCounter += 1
                if (opponent.history[i-1] == D):
                    adapReaction += 1
                elif (self.history[i-1] == D and opponent.history[i] == D):
                    adapReaction += 0.5
            elif (self.history[i-3] == D and self.history[i-2] == C):
                adapCounter += 1
                if (opponent.history[i-1] == C):
                    adapReaction += 1
                elif (self.history[i-1] == C and opponent.history[i] == C):
                    adapReaction += 0.5

        if adapCounter == 0:
            return 0
        
        return adapReaction/adapCounter*100

    
    @staticmethod
    def calc_forgiveness(self, opponent):
        DCounter = 0
        punishmentCounter = 0

        for i in range(0, len(self.history)-1):
            if(self.history[i] == D):
                DCounter += 1
                for j in range (i+1, len(opponent.history)):
                    if(opponent.history[j] == C):
                        break
                    else:
                        punishmentCounter += 1
              

        if punishmentCounter > 0:
            return DCounter/punishmentCounter*100
        else:
            return 100
    
    @staticmethod
    def calc_stochastic(self, opponent):
        patterns = [
            [C, C, C],
            [C, C, D],
            [C, D, C],
            [C, D, D],
            [D, C, C],
            [D, C, D],
            [D, D, C],
            [D, D, D]
        ]

        non_stochasticCounter = 0
        patternPlayedCounter = 0

        for p in patterns:
            opponentsReactions = []
            for i in range(0, len(self.history)-3):
                if ([self.history[i], self.history[i+1], self.history[i+2]] == p):
                    opponentsReactions.append([opponent.history[i+1], opponent.history[i+2], opponent.history[i+3]])
            
            unique_patterns = len(set(tuple(sub) for sub in opponentsReactions))

            if(len(opponentsReactions) > 0):
                non_stochasticCounter += 0 if unique_patterns == 1 else unique_patterns
                patternPlayedCounter += len(opponentsReactions)
        
        if patternPlayedCounter == 0:
            return 0
        
        return non_stochasticCounter/patternPlayedCounter*100

In [3]:
def build_player(params): 
    class FSOptimized(Player):

        name = "FuzzySugeno"
        classifier = {
            "memory_depth": float("inf"),
            "stochastic": False,
            "long_run_time": False,
            "inspects_source": False,
            "manipulates_source": False,
            "manipulates_state": False,
        }

        cooperation = ctrl.Antecedent(np.arange(0, 100, 1), 'cooperation')
        adaptivity = ctrl.Antecedent(np.arange(0, 100, 1), 'adaptivity')
        forgiveness = ctrl.Antecedent(np.arange(0, 100, 1), 'forgiveness')
        stochastic = ctrl.Antecedent(np.arange(0, 100, 1), 'stochastic')

        cooperation.automf(names=["low", "medium", "high"])
        adaptivity.automf(names=["no", "yes"])
        forgiveness.automf(names=["low", "medium", "high"])
        forgiveness['low'] = fuzz.gaussmf(forgiveness.universe, 0, 25)
        forgiveness['medium'] = fuzz.trimf(forgiveness.universe, [25, 50, 75])
        stochastic.automf(names=["none", "sometimes", "always"])

        resulting_strategy = ctrl.Consequent(np.arange(0, 100, 1),'resulting_strategy')
        resulting_strategy['D'] = fuzz.trimf(resulting_strategy.universe, [0, 25, 50])
        resulting_strategy['C'] = fuzz.trimf(resulting_strategy.universe, [35, 75, 100])

        rule1 = ctrl.Rule(cooperation['high'] & adaptivity['no'] & (forgiveness['medium'] | forgiveness['high']), resulting_strategy['D'])
        rule2 = ctrl.Rule(forgiveness['low'] & cooperation['high'], resulting_strategy['C'])
        rule3 = ctrl.Rule(stochastic['always'] | adaptivity['no'], resulting_strategy['D'])
        rule4 = ctrl.Rule(cooperation['low'] | (cooperation['medium'] & forgiveness['low']), resulting_strategy['D'])
        rule5 = ctrl.Rule(cooperation['medium'] & forgiveness['medium'] & adaptivity['yes'], resulting_strategy['C'])

        strategy_ctrl = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule5])
        chosen_strategy = ctrl.ControlSystemSimulation(strategy_ctrl)

        h = {'Name':'', 'Fuzzy' : [], 'Opponent': []}

        forgiveness_index = 0
        stochastic_index = 0
        first_time = True
    
        def strategy(self, opponent: Player) -> Action:

            if(len(self.history) == 0 or D not in opponent.history):
                return C
            
            else:
                coop = FuzzyMethods.calc_cooperation(self, opponent)
                adap = FuzzyMethods.calc_adaptivity(self, opponent)
                forg = FuzzyMethods.calc_forgiveness(self, opponent)
                stoch = FuzzyMethods.calc_stochastic(self, opponent)

                self.chosen_strategy.input['cooperation'] = coop
                self.chosen_strategy.input['adaptivity'] = adap
                self.chosen_strategy.input['forgiveness'] = forg
                self.chosen_strategy.input['stochastic'] = stoch
                
                self.chosen_strategy.compute()

                d_membership = fuzz.interp_membership(self.resulting_strategy.universe, self.resulting_strategy['D'].mf, self.chosen_strategy.output['resulting_strategy'])
                c_membership = fuzz.interp_membership(self.resulting_strategy.universe, self.resulting_strategy['C'].mf, self.chosen_strategy.output['resulting_strategy'])

                output = FuzzyMethods.fuzzy_gate(d_membership, params['d_threshold'], c_membership, params['c_threshold'])

                if(output > 0.5):
                    return D
            
            return C
    return FSOptimized()

In [4]:
i = 0
def fitness_function(ga_instance, solution, solution_idx):
    global i
    print(f"Evaluating solution {i} {solution_idx}: {solution}")
    if solution_idx == 7:
        i+=1
    results = axl.Tournament(players=[build_player(solution)] + [player() for player in axl.stewart_plotkin_strategies], turns=200).play(progress_bar=False)
    return np.mean(results.normalised_scores[0])


In [ ]:
j = 0
def objective(trial):
    global j
    d_threshold = trial.suggest_float('d_threshold', 0.1, 0.99)
    c_threshold = trial.suggest_float('c_threshold', 0.2, 0.9)

    params = {'d_threshold': d_threshold, 'c_threshold': c_threshold}
    try:
        fuzzy_player = build_player(params)
        j += 1
        return fitness_function(fuzzy_player, params, 0)

    except Exception as e:
        print(f"Failed to build player: {e}")
        return 0.0

In [ ]:
study = optuna.create_study(
    direction='maximize',
    study_name='fuzzy_optimization_sugeno1',
    storage='sqlite:///fuzzy_optimization_sugeno1.db',
    load_if_exists=True
)
study.enqueue_trial({
        'd_threshold': 0.4,
        'c_threshold': 0.6,
})
study.optimize(objective, n_trials=300, show_progress_bar=True)

[I 2026-05-23 00:46:06,890] A new study created in RDB with name: fuzzy_optimization_sugeno1
  0%|          | 0/300 [00:00<?, ?it/s]

Evaluating solution 0 0: {'d_threshold': 0.4, 'c_threshold': 0.6}


Best trial: 0. Best value: 2.72575:   0%|          | 1/300 [00:23<1:56:28, 23.37s/it]

[I 2026-05-23 00:46:30,267] Trial 0 finished with value: 2.7257500000000006 and parameters: {'d_threshold': 0.4, 'c_threshold': 0.6}. Best is trial 0 with value: 2.7257500000000006.
Evaluating solution 0 0: {'d_threshold': 0.3232540360725028, 'c_threshold': 0.8305148674343268}


Best trial: 0. Best value: 2.72575:   1%|          | 2/300 [00:47<1:58:13, 23.80s/it]

[I 2026-05-23 00:46:54,375] Trial 1 finished with value: 2.707571428571429 and parameters: {'d_threshold': 0.3232540360725028, 'c_threshold': 0.8305148674343268}. Best is trial 0 with value: 2.7257500000000006.
Evaluating solution 0 0: {'d_threshold': 0.3768009938335175, 'c_threshold': 0.5897436925229833}


... (592 outputs left out)

Best trial: 105. Best value: 2.76336: 100%|█████████▉| 299/300 [2:08:27<00:25, 25.35s/it]

[I 2026-05-23 02:54:34,348] Trial 298 finished with value: 2.707035714285714 and parameters: {'d_threshold': 0.6253993411661215, 'c_threshold': 0.5392619992926101}. Best is trial 105 with value: 2.7633571428571426.
Evaluating solution 0 0: {'d_threshold': 0.6510464303984768, 'c_threshold': 0.3101460701981501}


Best trial: 105. Best value: 2.76336: 100%|██████████| 300/300 [2:08:52<00:00, 25.78s/it]


[I 2026-05-23 02:54:59,543] Trial 299 finished with value: 2.716571428571428 and parameters: {'d_threshold': 0.6510464303984768, 'c_threshold': 0.3101460701981501}. Best is trial 105 with value: 2.7633571428571426.

=== OPTIMIZATION COMPLETE ===
Best score:  2.7634
Best params: {'d_threshold': 0.7593470762343023, 'c_threshold': 0.5192838614249861}

=== PARAMETER IMPORTANCE ===
  d_threshold: 0.9649
  c_threshold: 0.0351


In [7]:
print("\n=== OPTIMIZATION COMPLETE ===")
print(f"Best score:  {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

importance = optuna.importance.get_param_importances(study)
print("\n=== PARAMETER IMPORTANCE ===")
for param, imp in importance.items():
    print(f"  {param}: {imp:.4f}")


=== OPTIMIZATION COMPLETE ===
Best score:  2.7634
Best params: {'d_threshold': 0.7593470762343023, 'c_threshold': 0.5192838614249861}

=== PARAMETER IMPORTANCE ===
  d_threshold: 0.9677
  c_threshold: 0.0323
